# ⚡ Google Drive GPU Refiner - Remote Worker Node (Helper)

This lightweight helper notebook provides external GPU compute (NVIDIA Tesla T4 / V100 / A100 / L4) to accelerate video refining for the **Main Google Drive Downloader & GPU Refiner**.

---

### 🌟 How it Works:
1. **Shared Drive Integration**: Mounts the shared `downloader` Google Drive folder.
2. **Zero Configuration**: Automatically discovers the Main coordinator via `data/cluster/master.json` on Google Drive.
3. **Hardware NVENC Acceleration**: Receives 3–5 minute video chunks, transcodes them at maximum GPU speed using NVENC hardware acceleration, and saves the finished segments directly back into Google Drive.
4. **Fault Tolerant**: If this Colab runtime stops or is preempted, the Main coordinator automatically reclaims the chunk and reassigns it to another worker without losing any completed work.


In [ ]:
#@title 🛠️ Step 1: Environment Setup & Google Drive Mount
#@markdown Mounts Google Drive, verifies NVIDIA GPU hardware acceleration, and prepares workspace.

import os
import sys
import shutil
import subprocess
import json
import base64
import zlib

print("=" * 72)
print("🚀 [1/3] Checking Hardware Acceleration (NVIDIA GPU)...")
print("=" * 72)
has_gpu = False
try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        text=True
    ).strip()
    print(f"✅ Active GPU Detected: {gpu_info}")
    has_gpu = True
except Exception:
    print("⚠️ No NVIDIA GPU detected! Helper will run in CPU fallback mode (slower).")
    print("👉 Recommendation: Click Runtime > Change runtime type > Select T4 GPU.")

print("\n" + "=" * 72)
print("📁 [2/3] Mounting Google Drive...")
print("=" * 72)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully at /content/drive")
except Exception as e:
    print(f"⚠️ Drive mount notice: {e}")

print("\n" + "=" * 72)
print("📦 [3/3] Deploying Helper Modules...")
print("=" * 72)
workspace_dir = "/content/colab-gpu-helper"
os.makedirs(workspace_dir, exist_ok=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)

# Decompress and write modular files
bundle_b64 = "eJzdfYtW40iS6K/kULcHu9sYqNf2MMvMUuCuYptHLYaqnYNZHVlOGw2y5NEDysPy7zcemVKmlDKmunfPPdfdp7CtfERGRMY7048bQRJPw1l/sdzYExvhfJGkuUiyUazeprJ8WxThpPyQh/PqSXZb5GFUPbtNpT8J49konqbJXOTLBXwQ6un5Ig+T2I964iTM8p44CgP49yBeqtYLP7+NwrFu/hk+juJR/EocyVwGuRjE92GaxHMZ53viY5LMIikOk8gfi/tMnCSBD4AcD73D85ODD2If1tLHEfvyG8yWdUYb27DiHDqPNrrCjyciTvJmo8O9Eb42ujj1RE4BD1kS3UtvkjzEUeJPZOpNwrTTFVt/EVme7o1iAa/RBv1H74+WsT8PAZxoKSIAK5eZBvcoDe+lqIYSMBQsLUmXPZEVC1w3Iiy7hTdBkWc9HrGvv9jK/XQm82xrvNwKJwxbgVjt0YqyWz+VEzFNIhg7Ew+3Icw5K/zUh4VLHHkK32Q8KDQVZ4MvgwvxkIY5IEYBA1DniQ1wmiR5v7nQcEo41DhXqMDX2M8koskggz/O8G9Hf4angCbZ8TyEyfO63ap7KvMijcuuf0/CuKOH7AEIBi2y3J8hwzHBsPMrsdsXn9Nw7qdLEdzK4G6vFYEda5m6kVCNRBIzgynQ1NceogNWVnHU9gT7b7fMYqCrxm7mgF0DfXm6ND7ha5qksPMmIiSsRNAbmbCtfzmOh7MZRCBMmr16OGq32VORVvcLM5qOhnNNhC/ERRgX0gFGYzn6BbhLQ9ge++Ja9qPkQeLGwrXK+kp56hv3MACtHy87d9hJD4mj0BfXwC9+7o82kHNS/0G9kdMwlkCbm66Althd9mU8yR5CYNLRRj9cLOMxiooSHDVy2/rxpRiXoXU3y4pxgyDU3GZsYuiWtda4CEZkgWZTC79+HlRo1WwjvwVyAQKX/oB0WZviz3Rc+FlWbdPXfXGI21MEfpzEKDDx3SQEaslSMIZaXJWPiFuqcRu78HRJu3nbxGXvRe3Flhiqnby665DkLb3PrPm4j+JWZKAAGahagoGXJj2DOtnc+wdUbWRwEn4kARu4tzM1N7lEzYbfu9iHvm8hvAGzFus8DMoM3HL1p21IBy7vNYaqN4ZGN88zMgJgN1qDi5khW5bGeGhQw5gzqJj5TV8MpZ8Gt8K9UuICkDJKOYPp4LOuR3kjNivm2eQR50vq7lIzJe7a1Irq+7xGQYm2rElaR+eySxFF3qIuvVSHHo/mZj56pAV8n9auBa0l8lrI3K5AmBjujcDgtohRBdjvuSuMYf/f2R0Vklp2iX6tKfPx1dw1jqUzvhQNnt++3PA7NcnbcvOxRGZ7TsGYGULatZ1MId6+p8xRvm9jtY1Q9nPuLrPX77LFHEy9mkgtBNIvMCCaUHPjNSwapzUDsLjtmWeBpDbfyURHcuoXUS6m4LiNfcssqaw55Q1OVshlhyFg9GpKBn7ULRmvbPw/ISqcjWoi5OVr1KtUIoa749dH51/PTs4PjgYX3tHxBQzY5k2PYlh5zZE8OT88OPG+nl/8Ovx8cDiwwcnlfIFTyyiTqv2Hg+FATfNSp7M5lcXOemRk6DKIgB8Yii6HKYZ5WgSAAsCc5VYemabsxcHXGow0g42o0lOBkS8GvxyfDY7W7KS9GqPj6fmX48HQ1d8YGjvPk3sE0ex7+WWNfvm9l8m03vX88lNJ9VW9k/xWyYajg8uD9ZbJ/hx0OR1cHlC3fx+enzX6qfFobTL3sVf/71kSlxQbLG7lXKawwyl8JL6c2sRitmghWY1nFM28kIdXXWmt6/ZG4ulYxuXFwfCTN7w8+Hh89nHNAfLUz26tiAg3akOsY4gSs/xsJX7twd1YvhycflZT12YDel8cf9G4sWmswn6gD9JwXOS4oT5ficOoyHIwnf8IbmMR32Ww4WilNs1WoKzJSCauDj9dnf06XIVy4xFJAoLCwJYaYT1MG70PT66Gl+7NYmI34PVbXXDIwYVzTmNU7P2QpHcytSc8PaA/vxyfNERerffcx6kruqoFtwPeZA4TfLv7ikU05qkvxW6wYkHOkerLqrOdZrlDipUXqY9WhGIV7+jq4uDy+PzMGw4OYbbXb3f6O6VtAUapmIdxgfGKBYxwJ5fTFFSPyOQMI9iik4BJMgehM/YjPw6kSKaCDZA8iUAm4Vd/FGDPKS3yy9VwII63zzWneicD1EqXx6eD86tLBcOfNAiv4C1MBcpqkolI+pkUkxJ8xrf3aXBwcflhcHBZG+TtOxjklTj10zvBqAbYphEoFg4LiluwtfOx9HOydd++yxhxX2Q6TmCik2RG2/KPYFKNi5k4S/JwCnYUTp6JS3gaSdj9gw9XH72T84/DPTFOEvTgUE1LDvL3ZzIH02RwdvDhBGRE2ZbV7i7asX9Ae2CHyDb8GxD1FBsQ6fcwML9qJ2VLoOocjOVZRXUQWxhiprRDtmdrcJ1fIHTg0oJksdQC7D+uBlcA4odVE/6jkIXsT8YGy7Z1a24cqzPCeqzC0j3hRw/+MhMFwB0RgMPhkRj+x0mYSwr1YyRfRiEwIsbUrDURP2GbLehJi5r6YQTWS/ZnQTr5IYRRcWR7mS5LzViLvTjLPjNa1YckZRxnMLuIZSCzDGP3RghQkHMwipHf2DBWqZaaRFeqGt6YRkbTElJBPdvMqRqWtktPVISwpH9TWcB3pmxxyLae09bsCcvKKD+WlkNPNGwB3agCrq6Aym9MmFql7ijW9r/tzgJjzv07CYTIOpMeE8FL7vYv00Iqy3mFi6Xdq1coPoMiTWUcLMFRn/uL2wT4DATEVxYvJ8ChObQ9PfhPT5NUwwfc8oafnJx/9YafB0jVZpsdlHk7wBqTkDNuV2cnx6fHl4MjDFPz7DmK061sIUGyawdEzarQXY23S/k/1QjsWwU0PCmTnP1yKR0X4IAgmM6j6TznSGcJiFQAG//uiSLGjYpaJzDQhRyfIdigJGA7p2FwF0kTemh8D8oQUL8WlPZClUC5DOcyKXJyuoeEnq9+HtxOkpm4KChhWC4OuPCirjBQ6xhNKjpdfroYDD+dnxx5Hz4TUnd2xI/w7+u3faIXfv71w3bm7mzNsfuaJrEQzbBcDC4vYFujBsYxj+MwD0ES/gQfwSNV6aU3IAlzDPXl6LLlmRakW3M5T0DWXPrZnbhMfSUJAekYKdvCZLeIUFzOExgXZBHmtnNoTFmIG/XeQxFqIR3UxR16tfjYC5IiRisCuJTn/RglY6VN4Gv0KNMkygx+4yeeH+QcA8YNN4rZu1vrYStAnNTOfMTSPyW5weQR4z+kN52J7aFqnwndgbMaaZJlW4vIz+HTHBYyBz0WjsMozJfETHZ61Z/KfNl3ZJBp8kb6d7RxH05kotvSrBhB6GfFuJNuXmOefvvHv+6NNv71L/99s9kjWHvUruvuMhplP1E7UTbso7G36HStEAY+ud4DlrtBCGHBhJ6u+AuyoUCVRk00NsdLFHUeu18TGclcdii+gFrdg41JsZMKu2juOOoGPtAw9WKBSxxVfAi1ZD1NMI4JCpoVPFVOWB3CGJQ+Po+0TytLNxe5I1v4AXZOs1xpJGgco5E6BxETY/4/hU2Bs4Q5DxIt++LyNgRLNpX30CSzYPtHAVsLuTgr5qQD2ksFmljB5KujEKPZsNtkkF98oMRzesuhPF16DF9FHIKt5SmumWqH+hHrX/r4z9tOt38rv13v/Xzz5D1qkNFspD3kAPpJIwBfhEtgkCyvG30OIA1gDBBfEf3RHreJniDRmeBU4kEGbtWN63T6SFUHlD0DMmuuK1BJ8R3zGA8+CbO7VblLY6BasBUaMlt1nJMpkrIscxkWwgcr0LQuQBbnHSDSNXEhb54bUFwpxlD2wOYoIq7x4e0piJgkth6bKHjaE4/yyQpRNyP7zeWuZtPm0h3tW5OLFSpc6HCGsmsjqP3BQioALzAuFjwr1X54II/8JakAMO/Kr/fKQq1raHCj7JSuQ2ANAz/OmpVCouMOGnMpBU0qwJ3BnU+et1WUFAQg80FtYvGUrktCb0QCGasCnarwSYRZVoCogsHIo5ndJrC9wD8Dva/EG1VGge4xBBuMOAffBnW9Q3i75VeFIZiTbTddReaogGIS1HO5xhgA8BqZXYdodCRo9WQuWViV2DiSvM/a8PbIr9A7BksmzgxSTrTnjNjQEUCTpsxmC93Tqt4A1YxWAzTupKON/2JX98fRCPij/+P/QS0NLY4/np1fDA4PhoNur7VrGXvE3shWq/rfGGkfhBb0WibTewpRiLGcog2tNsyebng8LZ1BsL8Bo0ibWx/oqh10pKkVAsXZs6CQpI7HKPcrnY1M4KIXykVv0qgRag0I6H5uZlGj1aUSbf/mJGUqv2UONU89V86D2Tk7rDADg7F6+Bex4xaNpaaultimoPVLqTOMwkyJ/jxJTyPP0aVSF4dM1xtxQbSZiHs/CicVEYk6hjQDxfqoxn0yygxLWiHFX5Z6WINk2MlJNNdsBtnaJ1tBO5qsnXr8+H+cfjhNr0Lp99DQWn07IbFZpevXNzTKWZWRwdWllcNIW50EX2VN8DgkSyijusNfWEUCSkQbc3L+3FG72Vps0yjYJMSW49R5qVm6hxBBS4THFthNIwieANXz4LbDULgKq1ykwgQjVjeT2iCSYA3UFBxZxBcO9dRgWXw5/atyZc32Gts/URinMRq4xHelgtVtbfZ2gH+oGhYL8ag6PWkNON6u7BiHR8bl0zp6e1HEvHilYLDKeJ5MCmiuKt+xCKCIicNCFep1RV2fsehgOlB/G1Eym8m0tcwfPAV3aT/WKVpfrC7ur5X090ojUkWYqPMWmmHgSSAPJKk/wwyHin6MYg++b8YtLnTgAgg/LmYeeFrjSE7KnAFKGNTOA/4ercSJLuTAPeZn+AX0xJRED2s6QBGDKMAkxAT9KBg5lQFwH84PGxcXco3rQOuX1nJzo2I9HkchP3qfjoeX5xd/w7jSzo6CvDTt3fZzGXmROa+kY63ENq95MWhOgL9Fb+9VboW6IgYpw+JTAEaAsaw4KLOiKzMOM9mY40dUdVhi3OB8uzEAj9BpWI29VqVqrp1pmhsq19gdbSA31walAArlblZ6W8xYfEylIb3w6Eo1H0zmWiW+1qwBQtqEmSKNO0pzQbZwxiwHq3LTBHwDBYNFC+WS1aGsmELzUMfBQzZ36Efgz2R4SsHH2E0GbIsRbjIwSXMg+yNLi47sz/ooZUyh1HVyisXIz3GK1RgosFAHdnBFHgYE/dxDCZLl/nxRPzCj8KGlTL98EycPVL+WTvFjZ7Txw6e9H073fhgaZ3I8FH2phyGnBObH1GTHxtHXcAGiGOttFzJGQSJGIzxBEWDAlcUXmOSg3zA1FglKbpI8BaSqXXU+FDqyaW8rm1U9lGkeuqoILMwCLP8ThxjFj+JP7+gDfb1+xoTWaIyby285hw9rUVlyaqlNhuYNOHdJAQLYL/IEsK/OIKUS1hEgDopYxkEyIYlyFYfwFlZ966d00iWSMz9YCoXUVUsGndHnqfoVfF3n82lUZLcde+UdNfUgpn/TNAE5ez6kN6tLKwF64DMwTsE0STvVNGj50sqolqPHAQv2sIt8uvWzKWgIQEQtAo2qBv70qbMEMReAZYVwZPtYj4N4w6q5/kTq5zVzo4GKcuj2hhZOXiiiInkvIy9bzscglemDK2gf3aNqpMf9YrGQqZ4No9n4DGXz8dkv56MNV9h9VLzeffOnUTGVO1ONOhmZfUnytnWe/PxmAv9OdiaNzmDOwCb5enBxxol8fHd89tGu/DbheO/vrIJjcHFxftG6iH95G7i7Da8ODwfDYXvHnXc1uU1f77x+zdWOtD/lPCSpbZChJ3J/pt7NMYs9k+oTSJEABCWalaXpQuE5V1ztEMMPWp/gNFijwo8OFijPMuU7wJYPyxTWuJhOJUxViQM0tEgfhKRi+IwgwUCP1U53RLtaZb4O41eroTEJAxZnlKGxmtlW82it6BK+cowPOZSHHc9aQqNHe6jRRtkayCrKo5PVcwISn9GbxmOgHHX0Z41HipT4WL1t9kZLGbujCiPFZYD8ZGgLw9bs+0TMjsOVw5ST2RTd77r1WcOlNfIiWXR2rFAFrMvjqhj0ah7h49MNG2fwVltkppxkGuRkA3KX7OlGPDYFUPdJPKrR4Z1C0FNVFoyvV+Q+paBmZolWxqRzVayNWAkfxPKB2JbSTAbaXArfrYMruFnxxrWYDZqjtr1u49F91sSMczSKia3hus+HP2h7JQuksdUVJSKdU9S6bL/UX12MSkxbit+n/VVLN1uufSSKZVyMlVvLjluo2ZLrUzi73QIyJinmfmOj5EvM0R3NMV7eUfVKRG6J55dnfjqBpxlWv7Edza6ZbZ5WslarLZK0JTyWfNVYV6Z1EWA90VprGHLbNaFvB7FUL2tDyW7HOjAegfETohM1Cf1ZnGRg5NGG6WicJmjChhqblTfSDq4S2auAZVWloX3w03gtYHWcjHazjda1kamNhTUxSdbbWsCRvfmbQFPWx9qwgeFqSukO1fvsYcKea2PIinPFH5xuKI/E0lLleEBGZNUhhAkMYpvxrXpdWTmYHrL0zvUWw3jTtgYydd0LqdeQaLgrKTUvojwkNYBFGMAnydSxqvVWUKW5nPA3TJhmzNXAw2jjjK2n2CpdhYGTdAKALyXG86quuAZdEFQK5dqx8bqasfWxvN6EbzZvtFLWn12auZxRGxDlCNoCwnFqqhoe0+fNG1tfw/eKcaGTpS8qyxcVCYefadqumVJOmZXtrUXKPjNsU8TlbYhBv+V65LSsGZpHxzNvZQQOjapzenlYk/IizotNaneZFGkUheN+Kv9RSCw9tb8mMVN1LcaLNEHdsWa01BEn5VtRRhVwOvClPnIc19yGs0XhhfE04dCKLTIschzH2UKCLBBcFiLA45884PUfyKNnX46Pjg/ogAW2srcbDm/b2qMNnJaqnwDzQGrox1WctaPy96k/9+ZjbLVjPaBwdcq99TxPrjADViTuG7jtU7rFg68XRd6x98P1aCO+D0EnbmXzkH3LrS2gXLrcAnj3Ed4e82KfCgF7DIWnSid1D5ZN+0F234uTW4nn0+BNEWN16sZNzepH0bdvV2nUysnwtcCDmCgbFvoZp1rodGiR97NFFGKpew9P4TWcAOoN1v++eFMPjwJprk1q3FAIDppf79w4m5YUwZaY3+DWuzf1LCE313Sqxn19s2YEyxAfOJgRX41B3Qaefw82DFolLZFWuhYiQxycfRmcHYpbeR9s375+/5YtY7zbBtlX3ZhDRg9sOLB1lFRoD12t5ilY9XQ6X8iZ4gc9G9K+IncP42wgAPaNYY4GX86uTk5c4hOB53WjaCeic2gKF1R78Cx+65U9QYSlTZ9IJvI2dMQTDgqwX5N5UmS0zS+oUjTVxdcxRgGrfFMWwkdfXazEwhbVoAQi3alQwVGYBQnuHHHqA9z3oa9vHjri5CRAFc4zTJXliXhtFmBTRafgs09qOzGSMxqmlExEdw5YpMpuwGIPTP1N9BGazLqwyITAEdJA5/nw/Ozw6uJicHapKuWxZlj7hhTTAXkK1p3XyWQ07Slu8sLJ6pIofGGHvhbIHKM05LPRDiuY8AlyodnH3sp9FXsEucDymRkSQ5HVk0uZRX71QEsX4q0zQKlpM9Bc5Xpg8uo9tJ/qM1XeYwmgPkr+1FL/2Bgd+cionWSeFJ1He5mbepVgbnQbY/B5LA+UbFsWzWquuNZKBtaacJW0l0VJnu2xkgTRV6nduqmNMz3u7NFsPbHLb55WDNpeam114aJHLD6v3YtFdpUOabPOR4rWT0sSmWunn7s9Ei+lPWSZb2bEwgFBM0pRbYOJ2t9eRQ4agoS1RRZjB2gj359kgvuJsfQDlRbnnIvao2YeypLVJK/V3VtWmRi+mjhynPpDPFmY1alC10HE5gnAbu+5OVvPG7bNvOIgpGP+avqaK0GFGQo3z5alui8Da71EpYpHUSYl/Y4QFLGOT6VJVHuHpxg605a7V4CfsBYSC3X4mF/FZ6uuvcJuVMsJBIH3tUsubvN80X6DDL6U7oSu/ZSFJd990Oyx5l0ozqJbnewnjEi8mAwNyEB6ecKVGkq5APB5kemjiqONcBJJG34VQ+FwfpCA34Gm6tY0lVIfHTF3mEMZrtxoilyNOHqpGdBEt/VGI+xdyvyybflNo63pN6xQfY1+hiNR71ZZtI1ejF7qRO+a6QDgOa88yVqL3Tdap3KGCXZAqtdo7Az0f4cYMw/kvVSMVUeiXyrC1pj1GRFmzN0uvlDptUqw52PuuVNl1Xs8UCy9cb0e9TVtHIupn9y1kjReKRh5ZFrtd0hHEomTYr7o4KYDWHp4UBEU+f7r7w/Q43syWlHyeSpOoYSLjCeLhHKNZNngrHutJo82Zy3t7o5A4kvFyikN6cfi0+XlZ/H5fHiJhuTHwaVQkDDyxmnykIGgYl+a0o7+fRLiyfqkmEwjtPXHeEZDHUJuSCkVrqtbh86wnW0hspYBsj/WOj89avRYRmyTCWElnpKS9tD40itqSFDG0RWseutgRjfFYKTjNPlnGEX+Np6t73wN8ahfJs4uxe5Of+fPAr54//bP4tv7t11M8Ubyqxz/Gubb7978S//Ne9H59dPl6UlPROGdFB/BYU264vAWLCq5Teci8T8x9Kd+GqouVjCmguogQL5iiHyYR4U2t3kHOLv85xb7iVvHR02FYHd4sj9iEgKxF3KpPJ1ybc5gILncJhntk64uy6g2WbO7IgMog0O2jbculwup68+aa2xQGNrZob7+hdpK8DVvnX0NY09Pt6/+1gAilq+NBh9JgsDnHmkNcPP3d9+RvEhltnCghC+BwngRai7Mq7/ecRVf42ucTJZ8r9ICb76bdMpilVVY47XTpinNtayDYzXLUsyIZx93OydOalXSCm5UDeD7e3judbShb7wrxVETElWeSiklcBoHJ58HpIfQg0TJ8oinPCbyCUM81b7FemFcLsDeLBl2SgPXYq4uTv631gLMGYN1Vh5Y+p2W01q0boDDucUaPFXHBiBrQtBi7mpDycO9oBzIynk0In340j4jd9Hhu8qSpf1UxpiIHw4+H5MTwJXVH8j+bSgNAMJb+Evk6v9vLFynlalIrPL0NRqXlViB3BUayeWVo4TWIEnSCV4OkpSRGkNPdvv9fi0llOm4lW11gCPlL8JtZRZuaybg81clLewal5QKNOkSc+UGapu9y6VElc1tFWrhC0ygcMYHKqru+kuF516NXF3n7tD1ATXcqVIAPGWwFBUgLaiDzbf5qKd/2vyDhTTjBpTa3A1BMdrgq6yY1XlavrRHl7H+WWRLcCYntCFql5eTyzfWO8IqeuGoVdMTJYTvl85nY3dzQK3a3RlAUTlNytic4y0LQbaegemUAGxM0j0LizSZpViEgaxRXTQEdiNivbHTq12uoEAr9PGp0eDa2vU3mosdhsy6PG64jj3RZPDVGFfta3zfq8IA9lZxN1fbpCp8fYH8r2TDn96IT3ope3yzmnj1aM64SeF6QNK3ze4TiAnr2SLIN3sC7M/uXn93+vRDV/ybsFtMFxm2gK7wzsVkdMLgWWmQ3Bk8zdlnrZwdwcnn3JfPSRSp9AW6pzHWApdZKc5PKCGDmYYG22mFyQ6GxRcVWH8tuWu/7nLayqqF4fQkz4nNWz9TNko9+rWuengH7hhAjReeKfrr0Q3aIw3FT2L3abt6SrlUj9M5yBzJVPx7Mjb7/z0ZQ29inE2jX5hHEkbchNbwbLPbZnOk8vkIm04MVbzQEzwtlaT0RLkG9RkscIZZndfhCl2swsNDj6Bu98QU+Bepi4zdLraKMd5bpNhFgYHCGhEeyDWE1m8yTXiN2JDfNRqU68Y25YdGMxsd2Nb+ptHBRBU2tz67zJR1ZapFSbdcXU+oybfq1jlg6Ya2BiYNFJuXSHkyyddhHmasPnXrNtALhJZP/ra6kLXiWPZrGcWKYXHFe7Uijh7e/JRrPkZubOfF6vPgmwzo2kGdTN1CKPBKQcydckIdrIo4o+MfePCjvDkOD+SW6OmZyozP0CzAfgRdE1SKOevx7TF84YLAC0yzW5imP7+7tyLSZl7WBTVencoHGZk2+4SRa75T1eROwxBGNnX2qXP0jUm+qRdMZ6qtopq+edVTNTfATo9PBs2ZE6rx9c4zxi05qWplbL+bWgWy49h3DQEtopzLG2t8fz0ENhGPilmebgR5tXtimBRpoLVZeWUBzYd0P+JbiR5rM7f6f2XRgb1mlfJt3IZTX5C5hzETCc/rkVpnnhIWGMaUay1X6D1WU9vgcmeu8lhraGSVZ8a2I9lTOsRSLqFXTVknmeO6mJfl5Yx7Y35rkqp6Rj+PdIgbHQmgbyNtHlqu7hMChxClgyEO6sf3bUr3KgJ37Zlf98WHIowm4pdfsMrGEEMgfecgSY05YAS6nowvTlPbVktZ4yGbzTE4ahYjBOm00Q2+8+79CHu8MQPv9x7CEDTaU7EKP+NZPvVfv38ntsWnwZdD+yaHzKPyJJT58JfrefSwVJBxgD2Mb6u+aL5RFRBWZTkqpGqiQ03V5LVyHAcj3Ht+OuNk1MZWsHfPyzELkKjeia5BUVebL96qL4N/cHtMuwIG67/B5HBv22eMwvE3wg+PnE7NgRsg3Mt0idxnCVrHfN+3eKMs639z8e++Z/EGX88nPK5VqIYHASahLmNcqr9UCFnuRbu40NpfdJUub6H6PRHzCQivHKuMcS33DLPR1ywetBpbw/zIGGlYklt+M+2A32KKyIvlzKc6mzzTtRJ30vunTBNXHx04UAQMF3Jvt9GulNXV1zc1GfWmL058cNpvtZBSJX4c+qEYRRWWULdCrh2Qe/3e38Vry/mnfwwrbFp6XpUhgb4WqRzLHEU/qjRdlCOFHhQFcttrrUTHNhIasT1cpl0a+ZnyFkDUnjpYbBY8fj7+PFhRCGmVS46LKXoH+7sWqjkBP6ZTfmg0uRLqWDboLQLlidW+ny4y5/d8wes+FfTax9DcP9bCN1vHhAF1RtcV3MdW+/SnWeSrX1i1yw2MyhSAc/97f9vIWCi5pB0eXtUM47DXuzcanpZEz2/+ZR8+PNtYF+F5xcosYqyCe+0Z0VhDFvGKVRiNk4c2fjJfMEMHm25ZnEi11piUXYEl2mW1UKgjCWy+nvXWmx3W8t6b3WC3UAe1c55tT0G5vZLRnm1PBC170Kfn+5TFOGa4sr3XUwvJ8LU6AEACtuEPtchWrTLAB9LL3+vvTClQKf6bb0LWz2ih7uuJSshsgQa8Zbcl+fLgww5YcfbflcijjndhFNVZ+WVOYTnFpKA0UAtWNIFW5gIbviBGqRFMfk5K7Q94K7f7etfKZ8IG9VvHjKf7+41rx1Ytmg/9t62s0rdTPkP5QpUprgF8PJgzQbe5tlw8ubUmrugHy8gFq2ralZPe8gu8RsxjxUHkWjxkxUnkugdXYrxXD6rY/q8K8hnOtaZZvRt4SR286Fvd992M4LUk3vj2hecpqDEnOo8aKkpAiNMPLzeFxNDH37uqwlQbNaPwXZ/jictGNLGW8bFj0UrSG9K7jt5eidPajO/7fOOYEZ3DuEWS4i8jGJdrElleFJZwGxxrBisUC7aEJV5YvWbskXqiUd9jTjSKkmShq2aNgKijVpbrB7gUn39xU+Alr7CN6Hwbkr0z97+J1/Sef5kFnnSbZWcT8HDwuqCsurMPX/yT2tbhg4Yv+ny9GpEB7ZMsknLReeO6w668oK9GPbyoXsXSjQxY86JHCig76NyysHKB6OW4Dzm0sEOj8bUi0o0KhTa7PZeOrmwE1+WR2NmKoOMkJWvsq79t5Zy/39rMPNRvmwajF368rK70x5+jzszCOb5vly5nrA/cv/ejAu/7c/sjfEutOf7Ktb6gRkC/DE7e6b+riwR3hMRgQ/ctjRjpMlgV2qClYHz1g3i94zIP9GuVndiwjY5h3D3xFWwzfZtnlQw2j6zVnWYHCt5YmR4QEpydbsgqUiSq4ElSyiakXEyyAEMS4w8IiXV2DkWY6sCErwut9uADuVygit//3FTEz2SH5c876jAgzq9OdeHxMo5jkIVsF9rUI/IrZoHnSmYfH+lxzCz5C8YB6PZWa/36Q1Vbhc9OP4gvFwen3RfNyKm0T/rI4oHOsqHiE4+bf5PZpgrYNk+90rH+zbNEdPAA9S/qJ0e7my9bs/G7OnrtznOOT4p9QRninQt+LJMii5Za/b1oTv5Nr5Ok9rteuGK+BvJIdNQvgXVp/WpA43JFXvzR8VA1P9VXrGR0e8n6WLDYumaJrSPJTIOLuNnecOZmK3FVBZPIynBd+0B2BuxuMFZnfNzHTZaGZWYdI+QrUzt8YmKfk+d1q6gnMKy638Evel2sVgbDLHYdj8g5klI3FNQi9E0SuYWUZ40dyiXp08CC7kCegkPtqJC7ujj5Tjup1kLbPq4zim4F+PwUq5lMC8MA/p3u7pTr1aUFh9Uy9SY0aihbIwaMVle1rDMe5tSiBuCOWkJwpd5M31harUEXdYAM/nccGUNbuTyd2AgYNxazYsc9MAx4sZtaZetQhiqt2xL61WYbvxKfGwUKfC2mj7d65pYjja+XWWogwX5XA80dRmwJlilNogBQt8KwIKtHywwM7u6Ul7XADvbUWeE5cIG60lddz8dXHxunqOv3EeOde1zxhutUVgCbBdXlAIY5op382m3HxnQKaCVj9607DPQWsF1VdbkDmlT2haK/yuU4ATV8jL+ylRYLM3y/SnlUNs5kV69JmSOhHoqvlC4yGEMMb4ucthL+Tld9OyjoVmwC9SOaSvPgPXRkm3ge1wp5RBfPK3NwdYrhBThP/xc5uZVU"
try:
    raw_json = zlib.decompress(base64.b64decode(bundle_b64.encode('ascii'))).decode('utf-8')
    files_dict = json.loads(raw_json)
    for fname, content in files_dict.items():
        fpath = os.path.join(workspace_dir, fname)
        with open(fpath, "w", encoding="utf-8") as f:
            f.write(content)
    print(f"✅ Deployed {len(files_dict)} helper modules into {workspace_dir}")
except Exception as e:
    print(f"⚠️ Notice deploying bundle: {e}")

print("🎉 Setup complete! Helper workspace ready.")


In [ ]:
#@title 🚀 Step 2: Start Autonomous GPU Worker Loop
#@markdown Launches the helper node. It auto-discovers the Main coordinator and begins encoding video chunks.
enable_debug_logs = True #@param {type:"boolean"}

import time
import logger
logger.set_debug(enable_debug_logs)
import helper_worker

print("=" * 72)
print("🟢 Launching Autonomous GPU Worker Node...")
print(f"   Debug Log Notifications: {'ENABLED (Verbose)' if enable_debug_logs else 'DISABLED (Quiet)'}")
print("   Keep this cell running to provide GPU compute.")
print("=" * 72 + "\n")

try:
    helper_worker.run_helper_main(enable_debug_logs=enable_debug_logs)
except KeyboardInterrupt:
    print("\n🛑 Worker stopped by user.")
